# 🐙 Octopus Cloud

Free cloud GPU for the heavy jobs your computer can't do: **training Octopus brains**, **3D models** and **images**.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Then run the cells you need from top to bottom (click ▶ on each one). Every result downloads to your computer, and you import it into Octopus:

- Brains: **Train Octopus → Import brain** (the `.gguf` file)
- 3D models: **Modelling → Import a 3D model** (the `.zip` file)

Free Colab sessions stop after a few hours or when idle, so download results as soon as they finish.

In [ ]:
#@title 1. Set up Octopus (run this first, about 1 minute)
import os, subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU. Go to Runtime → Change runtime type → T4 GPU, then run this cell again.")
print("GPU:", gpu)
if not os.path.exists("/content/octopus"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/physicstheory/octopus.git", "/content/octopus"], check=True)
os.environ["OCTOPUS_HOME"] = "/content/octopus_home"
sys.path.insert(0, "/content/octopus")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft>=0.17", "accelerate", "datasets", "trimesh", "diffusers"], check=True)
print("✅ Octopus is ready")

## 🧠 Train a brain

Upload your own examples (the file from **Train Octopus → Download my data**, or any `.jsonl`, `.csv` or `.txt`) and/or tick open datasets. Training the 1.5B model on about 2,000 examples takes roughly 15 to 30 minutes on a free T4.

In [ ]:
#@title 2. Choose what to train on
BRAIN_NAME = "octopus-v1" #@param {type:"string"}
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct" #@param ["Qwen/Qwen2.5-Coder-0.5B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct", "Qwen/Qwen2.5-Coder-3B-Instruct"]
UPLOAD_MY_DATA = True #@param {type:"boolean"}
HELPFUL_CHAT_NoRobots = True #@param {type:"boolean"}
GENERAL_KNOWLEDGE_Dolly = False #@param {type:"boolean"}
CODING_CodeAlpaca = True #@param {type:"boolean"}
PYTHON_CODING_18k = False #@param {type:"boolean"}
EXAMPLES_PER_OPEN_DATASET = 1500 #@param {type:"integer"}
HOW_HARD = "Balanced" #@param ["Quick", "Balanced", "Thorough"]

from octopus import data
chosen = []
if UPLOAD_MY_DATA:
    from google.colab import files
    print("Choose your file(s), or press Cancel to skip")
    try:
        up = files.upload()
    except Exception:
        up = {}
    for name, raw in up.items():
        ds = data.save_upload(name, raw.decode("utf-8", errors="replace"))
        chosen.append(ds["id"]); print(f"Added {ds['count']} examples from {name}")
for flag, key in [(HELPFUL_CHAT_NoRobots, "no_robots"), (GENERAL_KNOWLEDGE_Dolly, "dolly"),
                  (CODING_CodeAlpaca, "code_alpaca"), (PYTHON_CODING_18k, "python_18k")]:
    if flag:
        chosen.append("builtin:" + key)
if not chosen:
    raise SystemExit("Pick at least one dataset or upload a file.")
print("Training on:", chosen)

In [ ]:
#@title 3. Train (watch the loss go down)
import time
from octopus.engine import Engine
from octopus.trainer import Trainer, TrainConfig

epochs, rank = {"Quick": (1, 8), "Balanced": (2, 16), "Thorough": (3, 32)}[HOW_HARD]
trainer = Trainer(Engine())
trainer.start(TrainConfig(name=BRAIN_NAME, base=BASE_MODEL, datasets=chosen, epochs=epochs, lora_rank=rank,
                          batch_size=4, grad_accum=4, builtin_limit=EXAMPLES_PER_OPEN_DATASET))
seq, final = 0, None
while trainer.running or seq < len(trainer.events):
    for ev in trainer.wait_events(seq, 5):
        seq = ev["seq"] + 1
        if ev["t"] == "log":
            print(ev["msg"])
        elif ev["t"] == "status" and ev["status"] in ("done", "error", "stopped"):
            final = ev
if not final or final["status"] == "error":
    raise SystemExit("Training failed: " + str(final and final.get("error")))
BRAIN_DIR = f"/content/octopus_home/brains/{final['brain']}"
print("\n✅ Trained! Test answer:\n", final.get("sample", ""))

In [ ]:
#@title 4. Package the brain for Octopus and download it (about 5 minutes)
#@markdown Makes a `.gguf` file for Octopus on any computer (Ollama), plus a small `.zip` for NVIDIA setups.
import gc, os, shutil, subprocess, sys, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import files

gc.collect(); torch.cuda.empty_cache()
merged = "/content/merged"
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16)
model = PeftModel.from_pretrained(base, BRAIN_DIR).merge_and_unload()
model.save_pretrained(merged, safe_serialization=True)
AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(merged)
del model, base; gc.collect()

if not os.path.exists("/content/llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/llama.cpp/gguf-py", "sentencepiece", "mistral-common"], check=True)
name = os.path.basename(BRAIN_DIR)
gguf = f"/content/{name}.gguf"
subprocess.run([sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", merged, "--outfile", gguf, "--outtype", "q8_0"], check=True)
shutil.make_archive(f"/content/{name}-adapter", "zip", BRAIN_DIR)
print(f"✅ {gguf} ({os.path.getsize(gguf) / 1e9:.1f} GB). Import it in Octopus: Train Octopus → Import brain.")
files.download(gguf)

## 🧊 3D models (Hunyuan3D-2)

Text or picture → textured 3D model (GLB, OBJ, STL). The first run installs Hunyuan3D and builds its texture renderer, which takes about 10 minutes. After that each model takes a few minutes.

Hunyuan3D-2 is released under the Tencent Hunyuan 3D 2.0 Community License: free to use, but not in the EU, UK or South Korea, and products with over 1 million monthly users need a licence from Tencent.

In [ ]:
#@title 5. Install Hunyuan3D (once per session, about 10 minutes)
import os, subprocess, sys
if not os.path.exists("/content/Hunyuan3D-2"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Tencent/Hunyuan3D-2", "/content/Hunyuan3D-2"], check=True)
os.chdir("/content/Hunyuan3D-2")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
for ext in ["hy3dgen/texgen/custom_rasterizer", "hy3dgen/texgen/differentiable_renderer"]:
    print("Building", ext, "…")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "."], cwd=ext, capture_output=True, text=True)
    print("  ok" if r.returncode == 0 else "  failed, textures will be skipped:\n" + r.stderr[-1500:])
os.chdir("/content")
import site, importlib; importlib.invalidate_caches(); sys.path.insert(0, "/content/Hunyuan3D-2")
print("✅ Hunyuan3D installed")

In [ ]:
#@title 6. Make a 3D model
PROMPT = "A wooden treasure chest with iron bands and a gold lock" #@param {type:"string"}
USE_MY_PICTURE_INSTEAD = False #@param {type:"boolean"}
QUALITY = "fast" #@param ["fast", "quality"]
TEXTURES = True #@param {type:"boolean"}
GAME_READY_LOW_POLY = True #@param {type:"boolean"}

import base64, os, shutil
from IPython.display import HTML, display
from octopus import studio as st

studio = st.Studio(free_others=lambda: None)
image_id = None
if USE_MY_PICTURE_INSTEAD:
    from google.colab import files
    up = files.upload()
    name, raw = next(iter(up.items()))
    image_id = st.save_upload(name, "data:image/png;base64," + base64.b64encode(raw).decode())["id"]

asset = None
for ev in studio.make_model(PROMPT, image_id=image_id, preset=QUALITY, texture=TEXTURES, game_ready=GAME_READY_LOW_POLY):
    if ev["t"] == "progress":
        print(f"{(ev.get('pct') or 0):5.0f}%  {ev['stage']}")
    elif ev["t"] == "error":
        raise SystemExit("❌ " + ev["d"])
    elif ev["t"] == "done":
        asset = ev["asset"]

folder = f"/content/octopus_home/assets/{asset['id']}"
glb = open(f"{folder}/{asset['files']['glb']}", "rb").read()
display(HTML(f'''<script type="module" src="https://cdn.jsdelivr.net/npm/@google/model-viewer@3.5.0/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{base64.b64encode(glb).decode()}" camera-controls auto-rotate shadow-intensity="1"
 style="width:100%;height:420px;background:#111;border-radius:16px"></model-viewer>'''))
safe = "".join(c if c.isalnum() else "-" for c in (PROMPT or "model"))[:40].strip("-") or "model"
zip_path = shutil.make_archive(f"/content/{safe}", "zip", folder)
print(f"✅ {asset.get('stats')}. Downloading {os.path.basename(zip_path)}. Import it in Octopus: Modelling → Import a 3D model.")
from google.colab import files
files.download(zip_path)

## 🖼️ Images

In [ ]:
#@title 7. Make images
PROMPT = "A floating island city at sunset, fantasy concept art" #@param {type:"string"}
MODEL = "sdxl" #@param ["sdxl", "flux"]
SHAPE = "1:1" #@param ["1:1", "16:9", "9:16", "4:3"]
HOW_MANY = 2 #@param {type:"integer"}

from IPython.display import Image as Show, display
from octopus import studio as st
from google.colab import files
studio = st.Studio(free_others=lambda: None)
for ev in studio.make_image(PROMPT, MODEL, SHAPE, HOW_MANY):
    if ev["t"] == "error":
        raise SystemExit("❌ " + ev["d"])
    if ev["t"] == "done":
        asset = ev["asset"]
folder = f"/content/octopus_home/assets/{asset['id']}"
for name in asset["images"]:
    display(Show(f"{folder}/{name}", width=420))
    files.download(f"{folder}/{name}")